# Program-level cumulative fatigue damage

**Program:** `13999` · **Version:** `v58` · **Data:** `RAW_DATA/test/13999/v58`

This notebook walks through the **two-layer damage model** used in durability analysis:

| Layer | Tool | Question it answers |
|-------|------|---------------------|
| **1 — Signal damage** | [py-fatigue](https://owi-lab.github.io/py_fatigue) rainflow + Palmgren–Miner | How much damage does **one pass** through this measured load history cause? |
| **2 — Schedule scaling** | autodam `.sch` file (`repeats`, `weight`, `*multiplier`) | How much does that event contribute over the **full program life**? |

References:
- RSP conversion: `notebooks/rsp_to_csv_v3.ipynb`
- Cycle counting & Miner damage: `notebooks/py_fatigue_cycle_counting.ipynb` and [py-fatigue damage example](https://owi-lab.github.io/py_fatigue/user/examples/07-damage.html)
- Schedule parsing & pattern matching: `notebooks/durability_schedule_extraction.ipynb`

---

## Design decisions (resolved for this notebook)

1. **Single-pass damage** uses Palmgren–Miner (`get_pm`) with DNV GL S–N curve C in air — same constants as `py_fatigue_cycle_counting.ipynb` and `server/services/fatigue_damage.py`.
2. **Program damage** = `D_single × repeats × weight × global_multiplier` (linear scaling; valid for linear PM).
3. **Channels:** the 12 Inspect Damage channels, resolved from CSV `#TITLES` using the same substring rules as `server/services/damage_channels.py`.
4. **Schedule matching:** longest substring containment on the RSP/CSV filename stem (same as `durability_schedule_extraction.ipynb`).
5. **Test slice:** only 3 RSP files exist under `RAW_DATA/test/13999/v58` (4e1 ballast family). Full-program totals here are **partial** — they sum only events present in the test folder, not all 60 schedule entries.

## Step 0 — Configuration

Set program/version paths. The folder layout is `RAW_DATA/test/{program}/{version}/`.

In [ ]:
from __future__ import annotations

import io
import json
import re
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import py_fatigue as pf

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

PROGRAM = "13999"
VERSION = "v58"


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for d in [start, *start.parents]:
        if (d / "server" / "pyproject.toml").is_file():
            return d
    return start


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "RAW_DATA/test" / PROGRAM / VERSION

print(f"Repo root: {REPO_ROOT}")
print(f"Data dir:  {DATA_DIR}")
print(f"Exists:    {DATA_DIR.is_dir()}")

## Step 1 — Convert RSP files to tagged CSV

RSP binaries are converted to the tagged CSV format (`#HEADER`, `#TITLES`, `#UNITS`, `#DATA`) used elsewhere in this repo.

We **reuse** the conversion functions from `rsp_to_csv_v3.ipynb` so this notebook stays in sync with the canonical converter.

In [ ]:
def load_rsp_converter() -> None:
    """Exec the rsp_to_csv_v3 helper cell into the global namespace."""
    nb_path = REPO_ROOT / "notebooks/rsp_to_csv_v3.ipynb"
    nb = json.loads(nb_path.read_text(encoding="utf-8"))
    src = nb["cells"][1]["source"]
    if isinstance(src, list):
        src = "".join(src)
    demo_start = src.find("rsp_path = _find_repo_root()")
    exec(src[:demo_start], globals())


load_rsp_converter()

rsp_files = sorted(DATA_DIR.glob("*.rsp"))
if not rsp_files:
    raise FileNotFoundError(f"No .rsp files in {DATA_DIR}")

conversion_log = []
for rsp_path in rsp_files:
    csv_path = rsp_path.with_suffix(".csv")
    if not csv_path.exists() or csv_path.stat().st_mtime < rsp_path.stat().st_mtime:
        out, shape, channel_count, _ = rsp_to_tagged_csv(rsp_path, csv_path)
        conversion_log.append({"rsp": rsp_path.name, "csv": out.name, "rows": shape[0], "channels": channel_count, "action": "converted"})
    else:
        conversion_log.append({"rsp": rsp_path.name, "csv": csv_path.name, "action": "reused existing csv"})

pd.DataFrame(conversion_log)

## Step 2 — Load the durability schedule (`.sch`)

An autodam schedule lists event **glob patterns** with two scaling factors per entry:

- **`repeats`** — how many times that event runs over the program (e.g. 16, 6000).
- **`weight`** — ballast / mixing fraction (often 0.75 + 0.15 + 0.10 = 1.0 across ballast variants).

The header may also define `*multiplier` (global program scale, default 1.0).

These fields are **not** handled by py-fatigue — they scale damage **after** single-pass calculation.

In [ ]:
def parse_autodam_schedule(path: Path) -> dict:
    path = path.expanduser().resolve()
    schedule = {
        "file": str(path),
        "schedule_id": None,
        "multiplier": 1.0,
        "entries": [],
    }
    entry_re = re.compile(r"^\*([^*]+)\*\s+(\d+)\s+([\d.]+)\s*$")
    id_re = re.compile(r"^\*id\s+(.+)$", re.IGNORECASE)
    mult_re = re.compile(r"^\*multiplier\s+([\d.]+)\s*$", re.IGNORECASE)

    for raw_line in path.read_text(errors="replace").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if line.lower().startswith("*summary"):
            break
        if m := id_re.match(line):
            schedule["schedule_id"] = m.group(1).strip()
        elif m := mult_re.match(line):
            schedule["multiplier"] = float(m.group(1))
        elif m := entry_re.match(line):
            schedule["entries"].append(
                {
                    "pattern": m.group(1),
                    "repeats": int(m.group(2)),
                    "weight": float(m.group(3)),
                }
            )
    return schedule


sch_files = sorted(DATA_DIR.glob("*.sch"))
if not sch_files:
    raise FileNotFoundError(f"No .sch schedule in {DATA_DIR}")
if len(sch_files) > 1:
    print(f"Warning: multiple .sch files; using {sch_files[0].name}")

schedule = parse_autodam_schedule(sch_files[0])
schedule_df = pd.DataFrame(schedule["entries"])

print(f"Schedule file: {Path(schedule['file']).name}")
print(f"Schedule ID:   {schedule['schedule_id']}")
print(f"Multiplier:    {schedule['multiplier']}")
print(f"Entries:       {len(schedule_df)}")
schedule_df.head(8)

## Step 3 — Match each event file to a schedule row

Each CSV/RSP filename is matched to the **longest** schedule pattern contained in the filename stem.

Example: `mf4e3_100_bt1cc_...` matches pattern `mf4e3_100` (repeats=16, weight=0.15).

### Plot — schedule scale factors for matched events (Step 3a)

`repeats × weight × multiplier` is the factor applied to single-pass damage in Step 6.

In [ ]:
def match_schedule_pattern(stem: str, patterns: list[str]) -> str | None:
    matches = [p for p in patterns if p in stem]
    return max(matches, key=len) if matches else None


patterns = schedule_df["pattern"].tolist()
lookup = schedule_df.reset_index().rename(columns={"index": "schedule_sequence"})
lookup["schedule_sequence"] = lookup["schedule_sequence"] + 1

event_rows = []
for csv_path in sorted(DATA_DIR.glob("*.csv")):
    stem = csv_path.stem
    matched = match_schedule_pattern(stem, patterns)
    row = {"rsp_file_name": csv_path.name, "matched_pattern": matched}
    if matched:
        meta = lookup.loc[lookup["pattern"] == matched].iloc[0]
        row.update(
            {
                "repeats": int(meta["repeats"]),
                "weight": float(meta["weight"]),
                "schedule_sequence": int(meta["schedule_sequence"]),
            }
        )
    event_rows.append(row)

events_df = pd.DataFrame(event_rows)
events_df["global_multiplier"] = schedule["multiplier"]
events_df["schedule_scale"] = events_df["repeats"] * events_df["weight"] * events_df["global_multiplier"]

display(events_df)

fig, ax = plt.subplots(figsize=(7, 3.5))
labels = events_df["matched_pattern"].astype(str)
x = np.arange(len(labels))
ax.bar(x - 0.2, events_df["repeats"], width=0.4, label="repeats", color="#93c5fd")
ax.bar(x + 0.2, events_df["weight"], width=0.4, label="weight", color="#fca5a5")
ax2 = ax.twinx()
ax2.plot(x, events_df["schedule_scale"], "o-", color="#16a34a", label="repeats × weight × mult")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("repeats / weight")
ax2.set_ylabel("combined scale")
ax.set_title("Schedule factors for matched test events")
lines1, lab1 = ax.get_legend_handles_labels()
lines2, lab2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, lab1 + lab2, loc="upper right", fontsize=8)
fig.tight_layout()
plt.show()

## Step 4 — Single-pass signal damage (py-fatigue / Layer 1)

### What py-fatigue does

For each channel time series:

1. **Rainflow cycle counting** — `CycleCount.from_timeseries` extracts stress cycles (`count_cycle` = \(n_j\) per bin).
2. **S–N lookup** — each bin gets `cycles_to_failure` \(N_j\) from the material curve.
3. **Palmgren–Miner rule** ([docs](https://owi-lab.github.io/py_fatigue/user/examples/07-damage.html)):

\[
D_{\text{single}} = \sum_j \frac{n_j}{N_j}
\]

This is **damage for one execution** of the load history. Schedule `repeats` and `weight` are applied in the next step.

In [ ]:
SN_CURVE = pf.SNCurve(
    [3, 5],
    intercept=[12.592, 16.320],
    norm="DNVGL-RP-C203/2016",
    environment="Air",
    curve="C",
)

MEAN_BIN_WIDTH = 100.0
RANGE_BIN_WIDTH = 100.0
MIN_POINTS = 3


def parse_tagged_csv(filepath: Path) -> tuple[pd.DataFrame, list[str]]:
    lines = filepath.read_text(encoding="utf-8", errors="replace").strip().split("\n")
    headers: list[str] = []
    data_start = 0
    for i, line in enumerate(lines):
        stripped = line.strip()
        if stripped == "#DATA":
            data_start = i + 1
            break
        if stripped == "#TITLES" and i + 1 < len(lines):
            headers = [h.strip() for h in lines[i + 1].split(",")]
    df = pd.read_csv(
        io.StringIO("\n".join(lines[data_start:])),
        header=None,
        dtype=float,
        on_bad_lines="warn",
    )
    if headers and len(headers) == df.shape[1]:
        df.columns = headers
    return df, headers


def single_pass_damage(values: np.ndarray) -> float:
    arr = np.asarray(values, dtype=float)
    finite = arr[np.isfinite(arr)]
    if finite.size < MIN_POINTS:
        return float("nan")
    time = np.arange(finite.size, dtype=float)
    cc = pf.CycleCount.from_timeseries(
        finite,
        time=time,
        mean_bin_width=MEAN_BIN_WIDTH,
        range_bin_width=RANGE_BIN_WIDTH,
    )
    return float(sum(pf.damage.stress_life.get_pm(cycle_count=cc, sn_curve=SN_CURVE)))


# Demo on first event, first matched BJ X channel
demo_csv = sorted(DATA_DIR.glob("*.csv"))[0]
demo_df, demo_headers = parse_tagged_csv(demo_csv)
title_cols = [c for c in demo_df.columns if isinstance(c, str) and c.strip()]
print(f"Demo file: {demo_csv.name}")
print(f"Measurement columns ({len(title_cols)}): {title_cols[:4]} ...")

### Plot — measured load history (Step 4a)

Before cycle counting, inspect the raw channel signal. The [py-fatigue damage tutorial](https://owi-lab.github.io/py_fatigue/user/examples/07-damage.html) plots stress vs time the same way for a sinusoidal example; here we plot a slice of the measured RSP-derived channel.

In [ ]:
# Pick BJ Y column by title substring (full channel map comes in Step 5).
demo_col = next((c for c in title_cols if "LBJ" in str(c) and "Fy" in str(c)), None)
if demo_col:
    y = demo_df[demo_col].to_numpy(dtype=float)
    n_plot = min(5000, y.size)
    t = np.arange(n_plot, dtype=float)

    fig, ax = plt.subplots(figsize=(12, 3.5))
    ax.plot(t, y[:n_plot], linewidth=0.6, color="#2563eb")
    ax.set_title(f"Load history — {demo_col}")
    ax.set_xlabel("Sample index (proxy time)")
    ax.set_ylabel("Channel value")
    fig.tight_layout()
    plt.show()
else:
    print("No BJ Fy column found for demo plot.")

## Step 5 — Resolve Inspect Damage channels

The dashboard uses 12 canonical damage channels (BJ / Shock / Bushing F&R × X/Y/Z).
We map CSV `#TITLES` to those keys using the same component/axis heuristics as `server/services/damage_channels.py`.

In [ ]:
DAMAGE_CHANNEL_LABELS = {
    "bj_x_force": "BJ X Force",
    "bj_y_force": "BJ Y Force",
    "bj_z_force": "BJ Z Force",
    "shock_x_force": "Shock X Force",
    "shock_y_force": "Shock Y Force",
    "shock_z_force": "Shock Z Force",
    "bushing_f_x_momt": "Bushing F X Momt",
    "bushing_f_y_momt": "Bushing F Y Momt",
    "bushing_f_z_momt": "Bushing F Z Momt",
    "bushing_r_x_momt": "Bushing R X Momt",
    "bushing_r_y_momt": "Bushing R Y Momt",
    "bushing_r_z_momt": "Bushing R Z Momt",
}

DAMAGE_CHANNEL_PATTERNS: dict[str, tuple[tuple[str, ...], str]] = {
    "bj_x_force": (("balljoint", "balljnt", "lbj", "otrbj", "outerbj"), "x"),
    "bj_y_force": (("balljoint", "balljnt", "lbj", "otrbj", "outerbj"), "y"),
    "bj_z_force": (("balljoint", "balljnt", "lbj", "otrbj", "outerbj"), "z"),
    "shock_x_force": (("shock", "shk"), "x"),
    "shock_y_force": (("shock", "shk"), "y"),
    "shock_z_force": (("shock", "shk"), "z"),
    "bushing_f_x_momt": (("frontattachment", "frontbush", "frontbsh", "frontbushing", "lcabushingf", "front bush"), "x"),
    "bushing_f_y_momt": (("frontattachment", "frontbush", "frontbsh", "frontbushing", "lcabushingf", "front bush"), "y"),
    "bushing_f_z_momt": (("frontattachment", "frontbush", "frontbsh", "frontbushing", "lcabushingf", "front bush"), "z"),
    "bushing_r_x_momt": (("rearattachment", "rearbush", "rearbsh", "rearbushing", "lcabushingr", "rear bush"), "x"),
    "bushing_r_y_momt": (("rearattachment", "rearbush", "rearbsh", "rearbushing", "lcabushingr", "rear bush"), "y"),
    "bushing_r_z_momt": (("rearattachment", "rearbush", "rearbsh", "rearbushing", "lcabushingr", "rear bush"), "z"),
}


def _normalized(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", value.lower())


def _tokens(value: str) -> set[str]:
    return {token for token in re.split(r"[^a-z0-9]+", value.lower()) if token}


def resolve_damage_columns(title_columns: list[str]) -> dict[str, str]:
    """Return {channel_key: csv_column_title} for uniquely matched channels."""
    resolved: dict[str, str] = {}
    for key, (component_patterns, axis) in DAMAGE_CHANNEL_PATTERNS.items():
        matches = []
        for col in title_columns:
            col_text = str(col)
            norm = _normalized(col_text)
            if not any(p.replace(" ", "") in norm or p in norm for p in component_patterns):
                continue
            toks = _tokens(col_text)
            if axis in toks or f"f{axis}" in toks or f"-{axis}" in col_text.lower():
                matches.append(col_text)
        if len(matches) == 1:
            resolved[key] = matches[0]
    return resolved


demo_map = resolve_damage_columns(title_cols)
pd.DataFrame([{"channel_key": k, "label": DAMAGE_CHANNEL_LABELS[k], "csv_column": v} for k, v in demo_map.items()])

### Plot — rainflow cycle-count histogram (Step 5b)

After resolving channels, `CycleCount.plot_histogram()` visualises the rainflow matrix (mean stress vs stress range). See [CycleCount definition](https://owi-lab.github.io/py_fatigue/user/examples/04-cyclecount-definition.html) and [CycleCount plotting in the user guide](https://owi-lab.github.io/py_fatigue/user/examples/04-cyclecount-definition.html). Marker size/colour encodes cycle count \(n_j\) per bin — the inputs to Palmgren–Miner ([damage example](https://owi-lab.github.io/py_fatigue/user/examples/07-damage.html)).

In [ ]:
demo_channel_key = "bj_y_force"
demo_col = demo_map.get(demo_channel_key)
if demo_col:
    demo_values = demo_df[demo_col].to_numpy(dtype=float)
    demo_finite = demo_values[np.isfinite(demo_values)]
    demo_cc = pf.CycleCount.from_timeseries(
        demo_finite,
        time=np.arange(demo_finite.size, dtype=float),
        mean_bin_width=MEAN_BIN_WIDTH,
        range_bin_width=RANGE_BIN_WIDTH,
        name=DAMAGE_CHANNEL_LABELS[demo_channel_key],
    )

    fig, ax = plt.subplots(figsize=(8, 5))
    demo_cc.plot_histogram(
        fig=fig,
        ax=ax,
        plot_type="mean-range",
        marker="s",
        s=40,
        cmap=mpl.colormaps["coolwarm"],
    )
    ax.set_title(f"Rainflow histogram — {DAMAGE_CHANNEL_LABELS[demo_channel_key]}")
    fig.tight_layout()
    plt.show()

    print(demo_cc)  # summary stats from CycleCount
else:
    print("Skip histogram: demo channel not resolved.")

## Step 6 — Combine layers: program damage per event × channel

\[
D_{\text{program, event, channel}} = D_{\text{single}} \times \text{repeats} \times \text{weight} \times \text{global\_multiplier}
\]

Special cases:
- `repeats = 0` or `weight = 0` → zero scheduled contribution.
- Unmatched schedule pattern → row flagged; no program scaling applied.

In [ ]:
def schedule_scale(repeats: int | float, weight: float, multiplier: float) -> float:
    return float(repeats) * float(weight) * float(multiplier)


detail_rows: list[dict] = []

for _, event in events_df.iterrows():
    csv_path = DATA_DIR / event["rsp_file_name"]
    df, _headers = parse_tagged_csv(csv_path)
    title_cols = [c for c in df.columns if isinstance(c, str) and str(c).strip()]
    channel_map = resolve_damage_columns(title_cols)

    matched = pd.notna(event.get("matched_pattern")) and event.get("matched_pattern")
    scale = (
        schedule_scale(event["repeats"], event["weight"], event["global_multiplier"])
        if matched
        else float("nan")
    )

    for channel_key, col_name in channel_map.items():
        values = df[col_name].to_numpy()
        d_single = single_pass_damage(values)
        d_program = d_single * scale if matched and np.isfinite(d_single) else float("nan")
        detail_rows.append(
            {
                "program": PROGRAM,
                "version": VERSION,
                "rsp_file_name": event["rsp_file_name"],
                "matched_pattern": event.get("matched_pattern"),
                "repeats": event.get("repeats"),
                "weight": event.get("weight"),
                "schedule_scale": scale,
                "channel_key": channel_key,
                "channel_label": DAMAGE_CHANNEL_LABELS[channel_key],
                "csv_column": col_name,
                "d_single": d_single,
                "d_program": d_program,
            }
        )

detail_df = pd.DataFrame(detail_rows)
detail_df.sort_values(["rsp_file_name", "channel_key"]).head(12)

## Step 7 — Cumulative program damage per channel

Sum `d_program` across all events in this folder for each channel.

> **Note:** With only the 3-file test slice, these totals cover the **4e1 ballast family only**. A production run would iterate all RSP files for program 13999 / version v58.

### Plot — cumulative damage by channel (Step 7a)

Bar chart of partial program-level damage summed across the test events. Values are dimensionless Miner damage \(D\); compare channels and events relatively, not to an absolute failure threshold, unless calibrated for your S–N curve and units.

In [ ]:
cumulative_df = (
    detail_df.groupby(["channel_key", "channel_label"], as_index=False)
    .agg(
        cumulative_d_program=("d_program", "sum"),
        events_contributing=("d_program", lambda s: int(np.isfinite(s).sum())),
        max_d_single=("d_single", "max"),
    )
    .sort_values("cumulative_d_program", ascending=False)
)

total_program_damage = cumulative_df["cumulative_d_program"].sum()
cumulative_df["pct_of_total"] = 100.0 * cumulative_df["cumulative_d_program"] / total_program_damage

print(f"Program {PROGRAM} / {VERSION} — partial cumulative damage (test folder only)")
print(f"Schedule: {schedule['schedule_id']}")
print(f"Global multiplier: {schedule['multiplier']}")
print(f"Events processed: {events_df['rsp_file_name'].nunique()}")
print(f"Sum of channel cumulative damage: {total_program_damage:.6f}\n")

display(cumulative_df)

# Bar chart — top channels by cumulative program damage
plot_df = cumulative_df.sort_values("cumulative_d_program", ascending=True)
fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(plot_df))))
colors = plt.cm.Blues(np.linspace(0.35, 0.85, len(plot_df)))
ax.barh(plot_df["channel_label"], plot_df["cumulative_d_program"], color=colors)
ax.set_xlabel("Cumulative program damage  D_program")
ax.set_title(f"Partial cumulative damage — program {PROGRAM} / {VERSION}")
for i, (_, row) in enumerate(plot_df.iterrows()):
    ax.text(row["cumulative_d_program"], i, f"  {row['pct_of_total']:.1f}%", va="center", fontsize=8)
fig.tight_layout()
plt.show()

## Step 8 — Event contribution matrix (% of channel total)

Useful for comparing which ballast variant (0.75 / 0.15 / 0.10 weight) drives each channel.

### Plot — stacked contribution by ballast variant (Step 8a)

Each bar shows how the three test events split cumulative damage for one channel. Labels use the matched schedule pattern (e.g. `4e1`, `mf4e3_100`, `mf4e3_400`).

In [ ]:
pivot = detail_df.pivot_table(
    index="rsp_file_name",
    columns="channel_key",
    values="d_program",
    aggfunc="first",
)

pct_pivot = pivot.div(pivot.sum(axis=0), axis=1) * 100.0
print("Program damage by event (absolute)")
display(pivot.round(4))
print("Share of each channel total (%)")
display(pct_pivot.round(1))

# Stacked bar chart — top 6 channels by total program damage
top_channels = (
    cumulative_df.sort_values("cumulative_d_program", ascending=False)
    .head(6)["channel_key"]
    .tolist()
)
event_labels = [
    detail_df.loc[detail_df["rsp_file_name"] == idx, "matched_pattern"].iloc[0]
    for idx in pivot.index
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absolute stacked bars
bottom = np.zeros(len(top_channels))
x = np.arange(len(top_channels))
palette = plt.cm.Set2(np.linspace(0, 1, len(pivot.index)))
for i, (event_name, pattern_label) in enumerate(zip(pivot.index, event_labels)):
    vals = pivot.loc[event_name, top_channels].fillna(0).to_numpy()
    axes[0].bar(x, vals, bottom=bottom, label=f"{pattern_label}", color=palette[i])
    bottom += vals
axes[0].set_xticks(x)
axes[0].set_xticklabels([DAMAGE_CHANNEL_LABELS[k] for k in top_channels], rotation=35, ha="right")
axes[0].set_ylabel("D_program")
axes[0].set_title("Absolute damage by event (stacked)")
axes[0].legend(title="Pattern", fontsize=8)

# Percent share heatmap for all resolved channels
hm = pct_pivot.T.loc[
    [k for k in cumulative_df["channel_key"] if k in pct_pivot.columns]
]
im = axes[1].imshow(hm.to_numpy(), aspect="auto", cmap="YlOrRd", vmin=0, vmax=100)
axes[1].set_yticks(range(len(hm.index)))
axes[1].set_yticklabels([DAMAGE_CHANNEL_LABELS[k] for k in hm.index], fontsize=8)
axes[1].set_xticks(range(len(hm.columns)))
axes[1].set_xticklabels(
    [detail_df.loc[detail_df["rsp_file_name"] == c, "matched_pattern"].iloc[0] for c in hm.columns],
    fontsize=9,
)
axes[1].set_title("Event share of channel total (%)")
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04, label="%")
fig.tight_layout()
plt.show()

## Step 9 — Inspect rainflow bins & Miner damage per bin

This mirrors the [py-fatigue damage tutorial](https://owi-lab.github.io/py_fatigue/user/examples/07-damage.html) workflow:

1. `cc.to_df()` — rainflow bins with `count_cycle`, `mean_stress`, `stress_range`
2. `df.miner.damage(sn_curve)` — adds `cycles_to_failure` and `pm_damage` per bin
3. `sum(pm_damage)` must equal `get_pm(cc, sn_curve)`

### Plot — per-bin Palmgren–Miner contribution (Step 9a)

The bar chart shows which stress ranges contribute most to \(D_{\text{single}} = \sum n_j/N_j\) before schedule scaling.

In [ ]:
if detail_df.empty:
    print("No detail rows to inspect.")
else:
    sample = detail_df.dropna(subset=["d_single"]).sort_values("d_single", ascending=False).iloc[0]
    sample_df, _ = parse_tagged_csv(DATA_DIR / sample["rsp_file_name"])
    values = sample_df[sample["csv_column"]].to_numpy(dtype=float)
    finite = values[np.isfinite(values)]
    cc = pf.CycleCount.from_timeseries(
        finite,
        time=np.arange(finite.size, dtype=float),
        mean_bin_width=MEAN_BIN_WIDTH,
        range_bin_width=RANGE_BIN_WIDTH,
    )
    bins = cc.to_df()
    bins.miner.damage(SN_CURVE)
    d_single = float(bins["pm_damage"].sum())
    print(
        f"Sample: {sample['rsp_file_name']} / {sample['channel_label']}\n"
        f"get_pm sum:     {sum(pf.damage.stress_life.get_pm(cc, SN_CURVE)):.6f}\n"
        f"df pm_damage:   {d_single:.6f}\n"
        f"× schedule:    × {sample['schedule_scale']} → d_program = {sample['d_program']:.6f}"
    )
    display(bins[["count_cycle", "mean_stress", "stress_range", "cycles_to_failure", "pm_damage"]].head(10))

    # Top bins by pm_damage
    top_bins = bins.nlargest(15, "pm_damage")
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

    axes[0].bar(
        range(len(top_bins)),
        top_bins["pm_damage"],
        color="#dc2626",
        alpha=0.85,
    )
    axes[0].set_xticks(range(len(top_bins)))
    axes[0].set_xticklabels(top_bins["stress_range"].round(0).astype(int), rotation=45, ha="right")
    axes[0].set_xlabel("Stress range (top 15 bins)")
    axes[0].set_ylabel("pm_damage  (n_j / N_j)")
    axes[0].set_title("Palmgren–Miner damage per rainflow bin")

    axes[1].scatter(
        bins["stress_range"],
        bins["count_cycle"],
        c=bins["pm_damage"],
        cmap="hot",
        s=30,
        alpha=0.7,
    )
    axes[1].set_xlabel("Stress range")
    axes[1].set_ylabel("count_cycle  (n_j)")
    axes[1].set_title("Cycle count vs range (colour = pm_damage)")
    fig.suptitle(f"{sample['channel_label']} — {sample['matched_pattern']}", y=1.02)
    fig.tight_layout()
    plt.show()